# فصل ۱۴: دانش خارجی و معماری RAG (External Knowledge & RAG)
## بخش ۲: الگوریتم‌های بازیابی - درس دوم: بازیابی با خود-کوئری (Self-Querying Retriever)

### 🎯 هدف درسنامه:
در درسنامه قبل با الگوریتم MMR جهت ایجاد تنوع در کانتکست آشنا شدیم. اما یک چالش بنیادین در پایگاه‌های برداری وجود دارد:
**«اعداد، تاریخ‌ها و فیلترهای منطقی دقیق (Mathematical & Exact Filters) در فضای برداری چگونه مدیریت می‌شوند؟»**

دیتابیس‌های برداری در درک **مفاهیم معنایی** (عشق، ترس، فناوری، درام) فوق‌العاده‌اند؛ اما در درک **«سال > ۲۰۱۰»** یا **«قیمت < ۱۰۰ دلار»** به شدت ضعیف عمل می‌کنند. چرا؟
زیرا در فضای تعبیه برداری (Embedding Space)، عدد `۲۰۱۰` فقط یک توکن متنی است که شاید به `۲۰۱۱` نزدیک باشد، اما مفهوم ریاضیاتی **«بزرگ‌تر / کوچک‌تر / مساوی»** را ندارد!

در این درسنامه با تکنیک پیشرفته **خود-کوئری (Self-Querying)** آشنا می‌شویم که در آن از مدل زبانی (LLM) به عنوان یک مترجم هوشمند استفاده می‌شود تا ورودی طبیعی کاربر را به دو بخش تجزیه کند:
1. **عبارت جستجوی معنایی (Semantic Query):** برای جستجوی برداری در متن اسناد.
2. **فیلتر ساختاریافته متادیتا (Metadata Filter):** اعمال شروط دقیق ریاضی و منطقی (مانند `rating > 8.5` و `genre == 'science fiction'`).


### 🔄 دیاگرام سازوکار خود-کوئری (Self-Querying Architecture):

```mermaid
flowchart TD
    UserQuery["👤 ورودی کاربر: 'فیلم‌های علمی تخیلی با امتیاز بالای ۸.۵'"] --> LLM["🤖 Query Constructor LLM (جمنای)"]
    Catalog["📋 کاتالوگ متادیتا (AttributeInfo Schema)"] --> LLM
    LLM --> Parsed["📦 ساختار تفکیک‌شده (Structured Query)"]
    Parsed --> |عبارت معنایی| SQ["🔍 Semantic Query: 'science fiction'"]
    Parsed --> |فیلتر متادیتا| MF["🎯 Metadata Filter: rating > 8.5"]
    SQ --> VDB["🗄️ Chroma Vector Store"]
    MF --> VDB
    VDB --> Results["📄 نتایج فیلترشده دقیق (مثلاً جنگ ستارگان با امتیاز ۸.۷)"]
```


In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

# فعال‌سازی حالت آفلاین برای مدل امبدینگ لوکال
os.environ["HF_HUB_OFFLINE"] = "1"

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# بارگذاری مدل امبدینگ چندزبانه
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# ۱. مجموعه داده فیلم‌های سینمایی همراه با شناسنامه و متادیتای غنی
movie_records = [
    ("دایناسورها در پارکی زنده می‌شوند و پارک را به آشوب می‌کشند.", 
     {"genre": "science fiction", "year": 1993, "rating": 7.7, "director": "Steven Spielberg"}),
    ("اسباب‌بازی‌ها وقتی کسی در اتاق نیست زنده می‌شوند و ماجراجویی می‌کنند.", 
     {"genre": "animated", "year": 1995, "rating": 9.1, "director": "John Lasseter"}),
    ("سه مرد برای سرقت مسلحانه وارد یک بانک می‌شوند.", 
     {"genre": "comedy", "year": 2010, "rating": 6.5, "director": "Unknown"}),
    ("یک ربات زباله‌جمع‌کن تنها در زمین متروکه عاشق می‌شود.", 
     {"genre": "science fiction", "year": 2008, "rating": 8.5, "director": "Andrew Stanton"}),
    ("جنگ ستارگان و نبرد حماسی شوالیه‌های جدای در کهکشان دور.", 
     {"genre": "science fiction", "year": 1977, "rating": 8.7, "director": "George Lucas"}),
    ("ماجراجویی یک باستان‌شناس شجاع در معابد باستانی و خطرناک.", 
     {"genre": "action", "year": 1981, "rating": 8.4, "director": "Steven Spielberg"}),
]

# ۲. ایجاد پایگاه برداری Chroma با متریک فاصله‌ای کسینوسی
vector_db = Chroma.from_texts(
    texts=[doc[0] for doc in movie_records],
    metadatas=[doc[1] for doc in movie_records],
    embedding=embedding_model,
    collection_name="movies_knowledge_base",
    collection_metadata={"hnsw:space": "cosine"}
)

print(f"✅ پایگاه برداری با {len(movie_records)} رکورد فیلم همراه با متادیتا ساخته شد.")


Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2743.78it/s]


✅ پایگاه برداری با 6 رکورد فیلم همراه با متادیتا ساخته شد.


### ۱. تعریف شناسنامه و کاتالوگ متادیتا (`AttributeInfo`)

برای اینکه مدل زبانی بتواند فیلترهای متادیتا بسازد، باید اسکیما و ستون‌های موجود در پایگاه داده را به همراه نوع داده و مقادیر مجاز بشناسد.
این کار با شیء **`AttributeInfo`** انجام می‌شود:


In [2]:
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.ir import Comparison

from langchain_community.query_constructors.chroma import ChromaTranslator

# مترجم مقاوم Chroma جهت تبدیل خودکار نوع عملوندها به int/float برای فیلترهای عددی
class RobustChromaTranslator(ChromaTranslator):
    def visit_comparison(self, comparison: Comparison):
        val = comparison.value
        # در صورتی که عملگر مقایسه‌ای عددی باشد، مقدار به عدد کست می‌شود تا دیتابیس خطا ندهد
        if comparison.comparator.value in ["gt", "gte", "lt", "lte"]:
            try:
                val = float(val) if "." in str(val) else int(val)
            except (ValueError, TypeError):
                pass
        return {
            comparison.attribute: {
                self._format_func(comparison.comparator): val
            }
        }

# تعریف مشخصات ستون‌ها و فیلدهای متادیتا
metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie. One of ['science fiction', 'comedy', 'drama', 'thriller', 'romance', 'action', 'animated']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released in theaters",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating",
        description="A 1-10 rating score for the movie",
        type="float",
    ),
]

# توصیف کلی درباره ماهیت متن اسناد در پایگاه
document_content_description = "Brief summary of a movie plot"

print("✅ کاتالوگ متادیتا (AttributeInfo) و مترجم مقاوم Chroma با موفقیت تعریف شدند.")


✅ کاتالوگ متادیتا (AttributeInfo) و مترجم مقاوم Chroma با موفقیت تعریف شدند.


### ۲. راه‌اندازی `SelfQueryRetriever` با مدل زبانی جمنای (`gemini-flash-lite-latest`)

اکنون `SelfQueryRetriever` را با دمای صفر (`temperature=0`) و `RobustChromaTranslator` راه‌اندازی می‌کنیم تا استنتاج فیلترها کاملاً قطعی و منطبق بر گرامر کوئری کروما باشد:


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

# راه‌اندازی مدل جمنای با دمای صفر
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    api_key=os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY"),
    temperature=0
)

# ساخت بازیاب خود-کوئری
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vector_db,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    structured_query_translator=RobustChromaTranslator(),
    verbose=True  # نمایش پردازش داخلی مدل و ساختار فیلتر
)

print(f"✅ بازیاب Self-Query با موفقیت متصل به مدل '{llm.model}' شد.")


✅ بازیاب Self-Query با موفقیت متصل به مدل 'gemini-flash-lite-latest' شد.


### ۳. تست عملی سناریوی اول: فیلتر عددی و مقایسه‌ای (`rating > 8.5`)

> **پرسش کاربر:** *«فیلم‌های علمی تخیلی که امتیاز بالاتر از ۸.۵ دارند را نشان بده.»*

در این پرسش:
- فیلم *WALL-E* امتیاز ۸.۵ دارد (چون شرط **بزرگ‌تر** است، نباید انتخاب شود).
- فیلم *Star Wars* امتیاز ۸.۷ دارد (باید انتخاب شود).
- فیلم *Jurassic Park* امتیاز ۷.۷ دارد (نباید انتخاب شود).


In [4]:
query_1 = "I want some science fiction movies rated higher than 8.5"
results_1 = retriever.invoke(query_1)

print(f"🎯 تعداد نتایج بازیابی‌شده: {len(results_1)}")
print("=" * 60)
for doc in results_1:
    print(f"🎬 خلاصه: {doc.page_content}")
    print(f"📑 متادیتا: {doc.metadata}")
    print("-" * 50)


🎯 تعداد نتایج بازیابی‌شده: 1
🎬 خلاصه: جنگ ستارگان و نبرد حماسی شوالیه‌های جدای در کهکشان دور.
📑 متادیتا: {'year': 1977, 'director': 'George Lucas', 'rating': 8.7, 'genre': 'science fiction'}
--------------------------------------------------


### ۴. تست عملی سناریوی دوم: فیلتر براساس نام کارگردان (`director == Steven Spielberg`)

> **پرسش کاربر:** *«فیلم‌های ساخته شده توسط استیون اسپیلبرگ را به من معرفی کن.»*


In [5]:
query_2 = "Show me movies directed by Steven Spielberg"
results_2 = retriever.invoke(query_2)

print(f"🎯 تعداد نتایج اسپیلبرگ: {len(results_2)}")
print("=" * 60)
for doc in results_2:
    print(f"🎬 خلاصه: {doc.page_content}")
    print(f"📑 متادیتا: {doc.metadata}")
    print("-" * 50)


🎯 تعداد نتایج اسپیلبرگ: 2
🎬 خلاصه: ماجراجویی یک باستان‌شناس شجاع در معابد باستانی و خطرناک.
📑 متادیتا: {'year': 1981, 'genre': 'action', 'director': 'Steven Spielberg', 'rating': 8.4}
--------------------------------------------------
🎬 خلاصه: دایناسورها در پارکی زنده می‌شوند و پارک را به آشوب می‌کشند.
📑 متادیتا: {'rating': 7.7, 'year': 1993, 'director': 'Steven Spielberg', 'genre': 'science fiction'}
--------------------------------------------------


### ۵. مقایسه عینی: جستجوی شباهت ساده در برابر خود-کوئری (چرا Similarity شکست می‌خورد؟)

بیایید همان پرسش اول (*امتیاز بالای ۸.۵*) را به جستجوی شباهت ساده بدهیم تا ببینیم بدون فیلتر متادیتا چه فاجعه‌ای رخ می‌دهد:


In [6]:
# جستجوی شباهت سنتی روی همان کوئری
sim_retriever = vector_db.as_retriever(search_kwargs={"k": 3})
sim_results = sim_retriever.invoke(query_1)

print("⚠️ نتایج جستجوی شباهت ساده (Similarity Search):")
print("=" * 60)
for idx, doc in enumerate(sim_results, 1):
    print(f"[{idx}] امتیاز: {doc.metadata.get('rating')} | ژانر: {doc.metadata.get('genre')}")
    print(f"    خلاصه: {doc.page_content}")
print("=" * 60)
print("❌ نقص جستجوی شباهت ساده: فیلم‌هایی با امتیاز کمتر از ۸.۵ (مثل دایناسورها با ۷.۷) بازیابی شدند چون متوجه شرط عددی نشد!")


⚠️ نتایج جستجوی شباهت ساده (Similarity Search):
[1] امتیاز: 8.7 | ژانر: science fiction
    خلاصه: جنگ ستارگان و نبرد حماسی شوالیه‌های جدای در کهکشان دور.
[2] امتیاز: 9.1 | ژانر: animated
    خلاصه: اسباب‌بازی‌ها وقتی کسی در اتاق نیست زنده می‌شوند و ماجراجویی می‌کنند.
[3] امتیاز: 7.7 | ژانر: science fiction
    خلاصه: دایناسورها در پارکی زنده می‌شوند و پارک را به آشوب می‌کشند.
❌ نقص جستجوی شباهت ساده: فیلم‌هایی با امتیاز کمتر از ۸.۵ (مثل دایناسورها با ۷.۷) بازیابی شدند چون متوجه شرط عددی نشد!


### ۶. ساخت زنجیره کامل RAG مجهز به SelfQueryRetriever و مدل جمنای

اکنون خط لوله کامل RAG را با استفاده از LCEL متصل به بازیاب هوشمند خود-کوئری می‌سازیم:


In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

template = """Answer the question based only on the following movie catalog context:
{context}

Question: {question}
If no matching movie is found in the context, say "موردی مطابق با شرایط یافت نشد".
"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    formatted = []
    for d in docs:
        m = d.metadata
        formatted.append(
            f"عنوان و خلاصه: {d.page_content}\n"
            f"ژانر: {m.get('genre')} | سال: {m.get('year')} | امتیاز: {m.get('rating')} | کارگردان: {m.get('director')}"
        )
    return "\n\n".join(formatted)

rag_chain_self_query = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ زنجیره کامل RAG با بازیاب خود-کوئری کامپایل شد.")


✅ زنجیره کامل RAG با بازیاب خود-کوئری کامپایل شد.


In [8]:
user_question = "فیلم‌های انیمیشن قبل از سال ۲۰۰۰ چه هستند و چه امتیازی دارند؟"
response = rag_chain_self_query.invoke(user_question)

print(f"پرسش کاربر: {user_question}")
print("=" * 60)
print("💬 پاسخ نهایی مدل جمنای:")
print(response)
print("=" * 60)


پرسش کاربر: فیلم‌های انیمیشن قبل از سال ۲۰۰۰ چه هستند و چه امتیازی دارند؟
💬 پاسخ نهایی مدل جمنای:
بر اساس متن ارائه شده:
عنوان: اسباب‌بازی‌ها (سال 1995، که قبل از سال 2000 است)
امتیاز: 9.1


### ⚖️ مزایا، معایب و مصالحه (Trade-offs) در خود-کوئری:

| فاکتور | ارزیابی | توضیحات |
| :--- | :--- | :--- |
| **دقت در فیلترها (Accuracy)** | بسیار بالا ⭐⭐⭐⭐⭐ | امکان اعمال دقیق شروط `>`، `<`، `==`، `in` و `and/or` بدون کدنویسی دستی |
| **تاخیر زمانی (Latency)** | متوسط به بالا ⏱️ | نیاز به یک مرحله پردازش اضافی LLM پیش از اجرای جستجو در دیتابیس |
| **وابستگی به اسکیما (Schema Dependency)** | حساس 📋 | اگر کاربر صفتی بگوید که در `AttributeInfo` نباشد، مدل ممکن است دچار سردرگمی شود |
| **هزینه توکن (Cost)** | اضافه بر جستجوی عادی 💰 | مصرف توکن برای تجزیه کوئری توسط پرامپت داخلی Query Constructor |

#### 🏢 بهترین موارد کاربرد در صنعت:
1. **فروشگاه‌های آنلاین و تجارت الکترونیک (E-Commerce):** فیلتر محدوده قیمت، برند کالا، وضعیت موجودی و امتیاز خریداران.
2. **پایگاه‌های حقوقی و اداری:** فیلتر براساس سال تصویب قانون، مرجع صادرکننده، شماره بخشنامه و دسته‌بندی موضوعی.
3. **آرشیو مقالات علمی و خبری:** فیلتر براساس نویسنده، ژورنال، سال انتشار و کلیدواژه‌ها.
